In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning\\research'

In [3]:
# Changing working directory one path back

os.chdir("../")

In [5]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int         #Exclude epochs and batch size

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path           # ResNet50 encoder saved here
    updated_base_model_path: Path   # Full encoder+decoder model saved here

    # ResNet50 Encoder params
    params_image_size: list         # e.g. [224, 224, 3]
    params_weights: str             # "imagenet"
    params_include_top: bool        # False (we remove classification head)
    params_pooling: str

    # LSTM Decoder params
    params_vocab_size: int          # total unique words in captions
    params_max_length: int          # max caption length (e.g. 35)
    params_embedding_dim: int       # word embedding size (e.g. 256)
    params_units: int               # LSTM hidden units (e.g. 512)
    params_dropout: float           # dropout rate (e.g. 0.5)

    # Training params (used later but defined here for MLflow tracking)
    params_learning_rate: float

In [8]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        
        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),

            # ResNet50 params (replacing VGG16 ones)
            params_image_size=self.params.IMAGE_SIZE,       # kept same
            params_weights=self.params.WEIGHTS,             # kept same
            params_include_top=self.params.INCLUDE_TOP,     # kept same
            params_pooling=self.params.POOLING,             # replaces CLASSES

            # LSTM Decoder params (new)
            params_vocab_size=self.params.VOCAB_SIZE,
            params_max_length=self.params.MAX_LENGTH,
            params_embedding_dim=self.params.EMBEDDING_DIM,
            params_units=self.params.UNITS,
            params_dropout=self.params.DROPOUT,
            params_learning_rate=self.params.LEARNING_RATE,
        )

        return prepare_base_model_config

In [10]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, LSTM, Embedding,
    Dropout, concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [11]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.resnet50.ResNet50(
            input_shape=self.config.params_image_size,  # (224, 224, 3)
            weights=self.config.params_weights,                 # "imagenet"
            include_top=self.config.params_include_top,         # False
            pooling=self.config.params_pooling                  # "avg" → 2048-dim
        )

        # Freeze ResNet50 — we don't retrain it
        self.model.trainable = False

        self.save_model(path=self.config.base_model_path, model=self.model)
        # logger.info(f"Base ResNet50 saved at: {self.config.base_model_path}")

    @staticmethod
    def _prepare_full_model(
        cnn_output_dim,         # 2048 from ResNet50 avg pool
        vocab_size,
        max_caption_length,
        embedding_dim,
        lstm_units,
        dropout_rate,
        learning_rate
    ):
        # 1. Image Feature Extractor branch
        input_image = Input(shape=(cnn_output_dim,), name='Features_Input')
        fe1 = Dropout(dropout_rate)(input_image)
        fe2 = Dense(embedding_dim, activation='relu')(fe1)

        # 2. Sequence Processor branch
        input_caption = Input(shape=(max_caption_length,), name='Sequence_Input')
        se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(input_caption)
        se2 = Dropout(dropout_rate)(se1)
        se3 = LSTM(lstm_units)(se2)

        # 3. Merge both branches
        decoder1 = concatenate([fe2, se3])

        # 4. Dense bottleneck
        decoder2 = Dense(embedding_dim, activation='relu')(decoder1)
        decoder2 = Dropout(dropout_rate)(decoder2)

        # 5. Output layer
        outputs = Dense(vocab_size, activation='softmax', name='Output_Layer')(decoder2)

        # Final model
        full_model = Model(
            inputs=[input_image, input_caption],
            outputs=outputs,
            name='Image_Captioning'
        )

        full_model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        full_model.summary()
        return full_model

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            cnn_output_dim=2048,                               # ResNet50 avg pool always 2048
            vocab_size=self.config.params_vocab_size,
            max_caption_length=self.config.params_max_length,
            embedding_dim=self.config.params_embedding_dim,
            lstm_units=self.config.params_units,
            dropout_rate=self.config.params_dropout,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)
        # logger.info(f"Full caption model saved at: {self.config.updated_base_model_path}")

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

In [12]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-06-24 16:39:51,927: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-24 16:39:51,952: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-24 16:39:51,960: INFO: common: created directory at: artifacts]
[2026-06-24 16:39:51,960: INFO: common: created directory at: artifacts/prepare_base_model]
94765736/94765736 [==============================] - 81s 1us/step
[2026-06-24 16:41:18,782: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "Image_Captioning"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 Sequence_Input (InputLayer)    [(None, 35)]         0           []                               
                                                                      